# Mô hình đối chứng Facebook Prophet — đo trên tập Test niêm phong

**Dự án:** Tốt nghiệp - Energy Forecasting - **Nhóm thực hiện:** The Outliers

## 1. TỔNG QUAN VÀ MỤC TIÊU

Notebook này dựng mô hình đối chứng Prophet để tính **Forecast Skill Score** cho mô hình
LightGBM chính thức của dự án.

### Vì sao phải viết lại notebook này

Bản trước đặt `TY_LE_TRAIN = 0.8` rồi **tự cắt 80/20 bên trong file audit**. Nghĩa là
Prophet được chấm điểm trên 20% cuối chuỗi của riêng nó, còn LightGBM được chấm trên
tập test niêm phong — **hai tập dòng khác nhau**. Ghép hai con số đó lại thành Skill
Score là sai phép so sánh, dù kết quả có đẹp đến đâu.

### Điều kiện so sánh công bằng trong bản này

| Điều kiện | Cách làm |
|---|---|
| Cùng dữ liệu học | Prophet học trên tập **Development** (train + val) — đúng phần LightGBM được học |
| Cùng mốc dự báo | Prophet dự báo tại đúng các mốc thời gian mục tiêu $T+h$ của tập test |
| Cùng tập chấm điểm | Lấy thẳng từ `prediction_audit.parquet` — cùng tử số, cùng mẫu số |
| Cùng phạm vi | Cả giá trị tại $T$ lẫn nhãn tại $T+h$ đều phải là số đo thật |

Prophet **chỉ** học từ lịch sử sản lượng và chu kỳ ngày/tuần của chính nó, không được
đưa bất kỳ đặc trưng thời tiết nào vào. Đó là bản chất của một baseline chuỗi thời gian
thuần túy — đối lập có chủ đích với LightGBM có đầy đủ đặc trưng thời tiết.

### Tính tái lập

Prophet dùng chế độ ước lượng MAP (không lấy mẫu MCMC) nên với cùng dữ liệu đầu vào,
kết quả lặp lại giống nhau giữa các lần chạy. Notebook này dùng cùng một logic với
`srcs/05_machine_learning/pipeline/actions/baseline_prophet_test_set.py`, nên số ra ở
đây trùng với số của pipeline.

## 2. Import thư viện và khai báo tham số

In [2]:
import json
import time
import warnings
import logging
from pathlib import Path

warnings.filterwarnings('ignore')
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
logging.getLogger('prophet').setLevel(logging.WARNING)

import numpy as np
import pandas as pd
from prophet import Prophet

# ── Đường dẫn ──
DEV_PATH = '../../data/model/v5/05_selected/v5_development_selected.parquet'
AUDIT_PATH = '../../data/model/v5/07_final_test/prediction_audit.parquet'
TEST_PATH = '../../data/model/v5/05_selected/v5_test_selected.parquet'
# SUA 2026-08-22: KHONG ghim cung ten loss nua. Ban truoc quan quan la huber nen ghim
# 'huber' van dung; ban nay quan quan doi sang mae, ghim cung se tro nham mo hinh.
# Doc thang tu best_loss.json - dung cach notebook 08 dang lam.
_BEST = Path('../../data/model/v5/07_final_test/best_loss.json')
if _BEST.exists():
    _b1 = json.loads(_BEST.read_text(encoding='utf-8')).get('h1', {})
    _LOSS_VD = str(_b1.get('folder_name') or _b1.get('winning_loss') or '').strip().lower()
else:
    _LOSS_VD = ''
if not _LOSS_VD:
    raise FileNotFoundError(
        'Khong doc duoc quan quan tu best_loss.json. Hay chay notebook 07_final_test truoc.')
CAU_HINH_MODEL = f'../../data/model/v5/06_train/{_LOSS_VD}/h1/model_config.json'
OUTPUT_DIR = Path('../../data/model/v5/08_baseline_prophet_test')

# ── Tham số ──
HORIZONS = [1, 4]        # h1 = 15 phút, h4 = 60 phút
BUOC_PHUT = 15           # lưới thời gian 15 phút
MIN_DONG_MOI_SITE = 200  # dưới ngưỡng này Prophet không đủ dữ liệu để học
# K_TARGET_MAX: dat o duoi, sau khi doc model_config (xem chu thich o do)
TRAN_HE_SO = 1.02        # train.yaml: tran_cong_suat_he_so

# Tham số chuẩn hoá đọc thẳng từ model_config.json của LightGBM — KHÔNG viết lại
# bằng tay. Hai mô hình phải dùng đúng cùng một công thức thì so sánh mới có nghĩa.
_c = json.loads(Path(CAU_HINH_MODEL).read_text(encoding='utf-8'))
COT_QUY_MO = _c['cot_quy_mo']        # site_scale
COT_SIN_ELEV = _c['cot_sin_elev']    # 'sin_elevation_mt' - mau so lay tai T+h
# SUA 2026-08-21: mau so chuan hoa da doi sang goc cao Mat Troi tai MOC MUC TIEU T+h,
# nen model_config ghi ten cot la 'sin_elevation_mt'. NHUNG notebook nay TU tra sang
# T+h bang hau to _mt_h{h} o o duoi, va cac tep 05_selected chi co cot GOC. Neu de
# nguyen hau to _mt thi vua doc thieu cot (KeyError o o doc DEV/TEST) vua dich hai
# lan thanh T+2h. Vi vay lui ve TEN GOC ngay tai day.
if COT_SIN_ELEV.endswith('_mt'):
    COT_SIN_ELEV = COT_SIN_ELEV[:-3]
COT_TRAN = _c['cot_tran']            # tran_cong_suat
EPS_ELEV = float(_c.get('eps_elev', 0.05))
# SUA 2026-08-22: nguong cat lay tu clip_k cua chinh mo hinh, khong ghim 1.5.
# Quy uoc du an: 06_4 cham validation bang k_target_max (1,5); 07/07b cham TEST
# bang clip_k. Notebook nay so tren tap TEST nen phai dung clip_k - neu khong
# Prophet duoc tran rong hon LightGBM va phep so khong cung dieu kien.
K_TARGET_MAX = float(_c['clip_k'])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Đã import thư viện và khai báo tham số.')
print(f'- Chuẩn hoá theo : {COT_QUY_MO} × max({COT_SIN_ELEV}, {EPS_ELEV})')
print(f'- Trần công suất : {COT_TRAN} × {TRAN_HE_SO}')
print(f'- Ghi kết quả ra : {OUTPUT_DIR}')


Đã import thư viện và khai báo tham số.
- Chuẩn hoá theo : site_scale × max(sin_elevation, 0.05)
- Trần công suất : tran_cong_suat × 1.02
- Ghi kết quả ra : ../../data/model/v5/08_baseline_prophet_test


## 3. Đọc tập học và chuẩn hoá mục tiêu

**Prophet học trên cùng mục tiêu chuẩn hoá với LightGBM**, không học trên kWh thô:

$$k = \frac{\text{energy}}{\text{site\_scale} \times \max(\sin h,\ \varepsilon)}$$

### Vì sao phải làm thế

LightGBM không dự báo kWh trực tiếp — nó dự báo tỉ lệ $k$ rồi mới nhân ngược ra kWh.
Nhờ vậy phần **hình học mặt trời** được xử lý bằng vật lý, mô hình chỉ phải học phần
biến động còn lại, và ban đêm tự về 0 vì $\sin h \approx 0$.

Bản trước bắt Prophet dự báo **thẳng kWh thô**. Nó phải tự học cả nhịp mọc/lặn bằng
chuỗi Fourier trơn — thứ về nguyên tắc không dựng được cạnh sắc. Kết quả đo trên tập
test: lúc 18:30 trạm tắt hẳn (thực tế 0.000, LightGBM 0.000) nhưng Prophet vẫn báo
3.980 và còn "rò" tới 20:45.

Đó là **hai bài toán khác độ khó**, không phải hai mô hình cùng điều kiện. Cho một bên
công cụ vật lý rồi bắt bên kia làm tay không thì so sánh không còn ý nghĩa.

Cách làm này cũng là chuẩn của ngành: dự báo mặt trời làm trên **chỉ số chuẩn hoá**
chứ không trên công suất thô — xem Lauret et al. (2022), *"Solar Forecasts Based on the
Clear Sky Index or the Clearness Index"*, DOI 10.3390/solar2040026, chính là bài báo
cáo đã trích dẫn.

Chỉ giữ dòng **ban ngày có số đo thật** — đúng tập mà LightGBM được học. Ban đêm không
cần học nữa vì phép nhân ngược đã ép về 0.


In [3]:
df_dev = pd.read_parquet(
    DEV_PATH,
    columns=['site_id', 'timestamp', 'energy_generated_kwh', 'energy_source',
             'is_daylight', COT_QUY_MO, COT_SIN_ELEV]
)
print(f'Tổng số dòng tập Development: {len(df_dev):,}')

giu = (
    (df_dev['energy_source'] == 'measured')
    & df_dev['is_daylight'].fillna(False).astype(bool)
    & (df_dev[COT_SIN_ELEV] > EPS_ELEV)
    & (df_dev[COT_QUY_MO] > 0)
)
df_dev = df_dev[giu].copy()

# Chuẩn hoá mục tiêu — đúng công thức LightGBM dùng
df_dev['k'] = (df_dev['energy_generated_kwh']
               / (df_dev[COT_QUY_MO] * np.maximum(df_dev[COT_SIN_ELEV], EPS_ELEV)))
df_dev['k'] = df_dev['k'].clip(0.0, K_TARGET_MAX)
df_dev = df_dev.rename(columns={'timestamp': 'ds', 'k': 'y'})[['site_id', 'ds', 'y']]

print(f'Số dòng ban ngày đo thật dùng để học: {len(df_dev):,}')
print(f'Số trạm: {df_dev["site_id"].nunique()}')
print(f'Phân bố k: min={df_dev.y.min():.4f}  trung vị={df_dev.y.median():.4f}  '
      f'max={df_dev.y.max():.4f}')
display(df_dev.head(3))


Tổng số dòng tập Development: 2,273,970
Số dòng ban ngày đo thật dùng để học: 930,306
Số trạm: 42
Phân bố k: min=0.0020  trung vị=0.6560  max=1.3686


,site_id,ds,y
25,1,2020-01-01 06:30:00,0.257426
26,1,2020-01-01 06:45:00,0.366217
27,1,2020-01-01 07:00:00,0.431518


## 4. Đọc tập chấm điểm và xác định phạm vi

Phạm vi lấy đúng điều kiện của chỉ số công bố, cộng thêm một điều kiện nữa: **nhãn tại
$T+h$ cũng phải là số đo thật**. Chấm một dự báo bằng một nhãn do ETL bịa ra thì con số
không còn đo năng lực dự báo nữa.

In [4]:
df_audit = pd.read_parquet(AUDIT_PATH)
df_goc = pd.read_parquet(
    TEST_PATH,
    columns=['site_id', 'timestamp', 'energy_source', COT_QUY_MO, COT_SIN_ELEV, COT_TRAN]
).rename(columns={'energy_source': 'src_goc'})

df_audit = df_audit.merge(df_goc, on=['site_id', 'timestamp'], how='left')
df_audit = df_audit.sort_values(['site_id', 'timestamp']).reset_index(drop=True)

mask_cham = {}
for h in HORIZONS:
    # Nguồn của NHÃN tại T+h: dịch cột nguồn lên h bước trong từng trạm
    # SUA 2026-08-21: tra nguon cua NHAN theo MOC THOI GIAN, khong shift theo dong.
    # shift theo dong chi dung khi luoi khong thung; thung mot moc la no vo phai dong ke
    # tiep con song, tuc gan nham nguon cua mot thoi diem khac.
    _moc = df_audit[['site_id', 'timestamp']].copy()
    _moc['timestamp'] = _moc['timestamp'] + pd.Timedelta(minutes=15 * h)
    _moc['_i'] = df_audit.index
    _tra = df_audit[['site_id', 'timestamp', 'src_goc']].rename(columns={'src_goc': '_src_nhan'})
    src_nhan = (_moc.merge(_tra, on=['site_id', 'timestamp'], how='left')
                .set_index('_i')['_src_nhan'].reindex(df_audit.index))
    mask_cham[h] = (
        (df_audit['energy_source'] == 'measured')        # giá trị tại T là đo thật
        & (src_nhan == 'measured')                        # nhãn tại T+h cũng đo thật
        & df_audit['is_daylight'].fillna(False).astype(bool)
        & df_audit[f'y_true_h{h}'].notna()
        & df_audit[f'y_pred_h{h}'].notna()
    )
    # Đại lượng để nhân ngược phải lấy tại MỐC MỤC TIÊU T+h, không phải tại T.
    # sin_elevation là đại lượng thiên văn tính trước được, nên đứng ở T đã biết
    # chính xác giá trị tại T+h — không phải leakage.
    # SUA 2026-08-21: tra theo MOC THOI GIAN thay cho shift(-h) theo dong.
    _cot_mt = [c for c in (COT_SIN_ELEV, COT_QUY_MO, COT_TRAN) if c in df_audit.columns]
    _m = df_audit[['site_id', 'timestamp']].copy()
    _m['timestamp'] = _m['timestamp'] + pd.Timedelta(minutes=15 * h)
    _m['_i'] = df_audit.index
    _t = df_audit[['site_id', 'timestamp'] + _cot_mt].rename(
        columns={c: f'{c}_mt_h{h}' for c in _cot_mt})
    _kq = _m.merge(_t, on=['site_id', 'timestamp'], how='left').set_index('_i')
    for c in _cot_mt:
        df_audit[f'{c}_mt_h{h}'] = _kq[f'{c}_mt_h{h}'].reindex(df_audit.index)
    print(f'h{h}: {int(mask_cham[h].sum()):,} dòng được chấm điểm')

print(f'\nTổng số dòng trong audit: {len(df_audit):,}')
display(df_audit[['site_id', 'timestamp', 'energy_source', 'is_daylight']].head(3))


h1: 220,114 dòng được chấm điểm
h4: 202,908 dòng được chấm điểm

Tổng số dòng trong audit: 486,160


,site_id,timestamp,energy_source,is_daylight
0,1,2021-12-18 09:30:00,measured,True
1,1,2021-12-18 09:45:00,measured,True
2,1,2021-12-18 10:00:00,measured,True


## 5. Hàm huấn luyện Prophet cho 1 trạm

Prophet học xu hướng và chu kỳ ngày/tuần. Tắt `yearly_seasonality` vì tập dữ liệu chưa
đủ nhiều năm để ước lượng chu kỳ năm cho ra hồn. Dự báo bị chặn dưới ở 0 — sản lượng
điện mặt trời không thể âm.

In [5]:
def train_prophet_1_tram(dev_tram, moc_can_du_bao):
    """Học Prophet trên lịch sử 1 trạm rồi dự báo tại các mốc thời gian cần.

    Trả về Series: index = mốc thời gian, value = **k** (tỉ lệ chuẩn hoá), chưa
    phải kWh. Việc nhân ngược ra kWh do giai_chuan_hoa() lo.
    """
    if len(dev_tram) < MIN_DONG_MOI_SITE or len(moc_can_du_bao) == 0:
        return None

    model = Prophet(
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=False,
    )
    model.fit(dev_tram[['ds', 'y']].sort_values('ds'))

    du_bao = model.predict(pd.DataFrame({'ds': moc_can_du_bao}))
    return pd.Series(du_bao['yhat'].to_numpy(), index=moc_can_du_bao)


def giai_chuan_hoa(k, quy_mo, sin_elev, tran):
    """Nhân ngược k -> kWh, ĐÚNG y công thức LightGBM dùng ở s09b.

        y = clip(k, 0, 1.5) × site_scale × max(sin_elev, eps)
        y = min(y, tran_cong_suat × 1.02)
        y = 0  khi sin_elev <= eps      <- ban đêm tự về 0, không phải chặn tay
    """
    y = np.clip(k, 0.0, K_TARGET_MAX) * quy_mo * np.maximum(sin_elev, EPS_ELEV)
    y = np.minimum(y, tran * TRAN_HE_SO)
    return np.where(sin_elev <= EPS_ELEV, 0.0, y)


def tinh_wape(y_that, y_bao):
    """WAPE = tổng |sai số| / tổng |thực tế|, đơn vị %."""
    mau = np.abs(y_that).sum()
    return float(np.abs(y_that - y_bao).sum() / mau * 100.0) if mau > 0 else np.nan


print('Đã định nghĩa train_prophet_1_tram(), giai_chuan_hoa(), tinh_wape().')


Đã định nghĩa train_prophet_1_tram(), giai_chuan_hoa(), tinh_wape().


## 6. Chạy Prophet cho toàn bộ 42 trạm

Mỗi trạm mất khoảng 10–15 giây, tổng khoảng 8–10 phút.

In [6]:
t0 = time.time()
sites = sorted(df_audit['site_id'].unique())
ket_qua_tram, site_loi = [], []

for i, s in enumerate(sites, 1):
    dev_tram = df_dev[df_dev['site_id'] == s]
    audit_tram = df_audit[df_audit['site_id'] == s]

    # Dự báo tại MỌI mốc mục tiêu của trạm, KHÔNG chỉ các dòng được chấm điểm.
    # Nếu chỉ lấy mốc trong mask, cột prophet_h* sẽ rỗng ở mọi dòng ngoài phạm vi
    # chấm — vẽ lên dashboard thì đường Prophet đứt quãng từng đoạn. Chấm điểm và
    # vẽ là hai việc khác nhau: chấm phải lọc chặt, vẽ phải liền mạch.
    moc = pd.DatetimeIndex(sorted({
        t for h in HORIZONS
        for t in (audit_tram['timestamp'] + pd.Timedelta(minutes=BUOC_PHUT * h)).dropna()
    }))

    try:
        du_bao_k = train_prophet_1_tram(dev_tram, moc)
    except Exception as e:
        site_loi.append({'site_id': s, 'loi': str(e)[:150]})
        print(f'   [{i:>2}/{len(sites)}] trạm {s}: LỖI — {str(e)[:70]}')
        continue
    if du_bao_k is None:
        continue

    ghi = {'site_id': s, 'n_train': len(dev_tram)}
    for h in HORIZONS:
        moc_h = audit_tram['timestamp'] + pd.Timedelta(minutes=BUOC_PHUT * h)
        # Prophet trả về k; nhân ngược ra kWh bằng đại lượng tại mốc mục tiêu T+h
        k = du_bao_k.reindex(moc_h).to_numpy(float)
        df_audit.loc[audit_tram.index, f'prophet_h{h}'] = giai_chuan_hoa(
            k,
            audit_tram[f'{COT_QUY_MO}_mt_h{h}'].to_numpy(float),
            audit_tram[f'{COT_SIN_ELEV}_mt_h{h}'].to_numpy(float),
            audit_tram[f'{COT_TRAN}_mt_h{h}'].to_numpy(float),
        )

        # Chỉ số thì CHỈ tính trên các dòng trong phạm vi chấm điểm.
        idx = audit_tram.index[mask_cham[h][audit_tram.index]]
        if len(idx) == 0:
            continue
        y_that = df_audit.loc[idx, f'y_true_h{h}'].to_numpy(float)
        y_bao = df_audit.loc[idx, f'prophet_h{h}'].to_numpy(float)
        ok = ~np.isnan(y_bao)
        ghi[f'n_test_h{h}'] = int(ok.sum())
        ghi[f'wape_prophet_h{h}'] = round(tinh_wape(y_that[ok], y_bao[ok]), 4)
        ghi[f'wape_model_h{h}'] = round(
            tinh_wape(y_that[ok], df_audit.loc[idx, f'y_pred_h{h}'].to_numpy(float)[ok]), 4)
    ket_qua_tram.append(ghi)

    mo_ta = '  '.join(
        f"h{h}: Prophet {ghi.get(f'wape_prophet_h{h}', np.nan):.2f}% / "
        f"model {ghi.get(f'wape_model_h{h}', np.nan):.2f}%" for h in HORIZONS)
    print(f'   [{i:>2}/{len(sites)}] trạm {s}: {mo_ta}   ({(time.time()-t0)/60:.1f} phút)')

df_ket_qua = pd.DataFrame(ket_qua_tram)
print(f'\nĐã chạy xong {len(df_ket_qua)}/{len(sites)} trạm trong {(time.time()-t0)/60:.1f} phút.')
if site_loi:
    print(f'Số trạm lỗi: {len(site_loi)} — {[x["site_id"] for x in site_loi]}')
display(df_ket_qua.head(5))


01:10:23 - cmdstanpy - INFO - Chain [1] start processing
01:10:26 - cmdstanpy - INFO - Chain [1] done processing


   [ 1/40] trạm 1: h1: Prophet 27.05% / model 13.12%  h4: Prophet 26.75% / model 15.83%   (0.1 phút)


01:10:27 - cmdstanpy - INFO - Chain [1] start processing
01:10:32 - cmdstanpy - INFO - Chain [1] done processing


   [ 2/40] trạm 2: h1: Prophet 28.06% / model 13.39%  h4: Prophet 27.72% / model 16.14%   (0.2 phút)


01:10:34 - cmdstanpy - INFO - Chain [1] start processing
01:10:39 - cmdstanpy - INFO - Chain [1] done processing


...

01:12:51 - cmdstanpy - INFO - Chain [1] start processing
01:12:54 - cmdstanpy - INFO - Chain [1] done processing


   [39/40] trạm 41: h1: Prophet 31.46% / model 21.89%  h4: Prophet 31.30% / model 27.63%   (2.6 phút)


01:12:56 - cmdstanpy - INFO - Chain [1] start processing
01:13:01 - cmdstanpy - INFO - Chain [1] done processing


   [40/40] trạm 42: h1: Prophet 25.58% / model 13.20%  h4: Prophet 25.18% / model 15.76%   (2.7 phút)

Đã chạy xong 40/40 trạm trong 2.7 phút.


,site_id,n_train,n_test_h1,wape_prophet_h1,wape_model_h1,n_test_h4,wape_prophet_h4,wape_model_h4
0,1,29992,5888,27.0489,13.1188,5457,26.7481,15.8263
1,2,29439,5868,28.0609,13.3882,5431,27.7210,16.1448
2,3,29728,5782,31.1627,15.0905,5355,30.4765,17.5976
3,4,14002,4914,27.2871,16.6701,4657,27.0618,18.7023
4,5,13797,3761,28.2495,17.2610,3540,27.7220,19.0862


## 7. Kết quả tổng hợp và Forecast Skill Score

WAPE tổng hợp bằng cách **gộp toàn bộ sai số tuyệt đối rồi chia tổng sản lượng thật**,
không lấy trung bình WAPE theo trạm. Trung bình của tỷ số không bằng tỷ số của tổng —
lấy trung bình theo trạm sẽ cho trạm nhỏ cùng trọng số với trạm lớn, làm lệch con số.

In [7]:
def bo_chi_so(y_that, y_bao):
    """Bộ chỉ số đầy đủ cho 1 cặp (thực tế, dự báo)."""
    e = y_bao - y_that
    return {
        'wape': np.abs(e).sum() / np.abs(y_that).sum() * 100,
        'rmse': np.sqrt((e ** 2).mean()),
        'mae': np.abs(e).mean(),
        'r2': 1 - (e ** 2).sum() / ((y_that - y_that.mean()) ** 2).sum(),
    }


tom_tat = {
    'nguon_hoc': 'v5_development_selected (train+val), chỉ dòng measured',
    'nguon_cham_diem': '07_final_test/prediction_audit.parquet',
    'so_site': int(len(df_ket_qua)),
    'so_site_loi': len(site_loi),
}
bang_bao_cao = []

for h in HORIZONS:
    m = mask_cham[h] & df_audit[f'prophet_h{h}'].notna()
    y_that = df_audit.loc[m, f'y_true_h{h}'].to_numpy(float)
    cs_model = bo_chi_so(y_that, df_audit.loc[m, f'y_pred_h{h}'].to_numpy(float))
    cs_prophet = bo_chi_so(y_that, df_audit.loc[m, f'prophet_h{h}'].to_numpy(float))
    ss = (1 - cs_model['wape'] / cs_prophet['wape']) * 100

    tom_tat[f'h{h}'] = {
        'n_dong': int(m.sum()),
        'wape_prophet_%': round(cs_prophet['wape'], 4),
        'wape_lightgbm_%': round(cs_model['wape'], 4),
        'skill_score_%': round(ss, 4),
    }
    for ten, cs in (('LightGBM (đủ đặc trưng thời tiết)', cs_model),
                    ('Prophet (không đặc trưng thời tiết)', cs_prophet)):
        bang_bao_cao.append({
            'horizon': f'h{h}', 'mo_hinh': ten, 'n_dong': int(m.sum()),
            'WAPE_%': round(cs['wape'], 4), 'RMSE': round(cs['rmse'], 4),
            'MAE': round(cs['mae'], 4), 'R2': round(cs['r2'], 4),
        })

    print(f'=== h{h} — {int(m.sum()):,} dòng, CÙNG tập dòng cho cả hai mô hình ===')
    print(f'   Prophet  WAPE = {cs_prophet["wape"]:7.4f}%')
    print(f'   LightGBM WAPE = {cs_model["wape"]:7.4f}%')
    print(f'   Forecast Skill Score = {ss:+7.2f}%\n')

df_bao_cao = pd.DataFrame(bang_bao_cao)
display(df_bao_cao)

=== h1 — 220,114 dòng, CÙNG tập dòng cho cả hai mô hình ===
   Prophet  WAPE = 35.0998%
   LightGBM WAPE = 17.5197%
   Forecast Skill Score =  +50.09%

=== h4 — 202,908 dòng, CÙNG tập dòng cho cả hai mô hình ===
   Prophet  WAPE = 34.9435%
   LightGBM WAPE = 21.7353%
   Forecast Skill Score =  +37.80%



,horizon,mo_hinh,n_dong,WAPE_%,RMSE,MAE,R2
0,h1,LightGBM (đủ đặc trưng thời tiết),220114,17.5197,3.3043,1.3803,0.9293
1,h1,Prophet (không đặc trưng thời tiết),220114,35.0998,5.4661,2.7653,0.8064
2,h4,LightGBM (đủ đặc trưng thời tiết),202908,21.7353,4.0851,1.7967,0.8953
3,h4,Prophet (không đặc trưng thời tiết),202908,34.9435,5.6137,2.8886,0.8022


### Nhận xét

Prophet chỉ học chu kỳ ngày/tuần từ lịch sử sản lượng, không có đặc trưng thời tiết thật
(`shortwave_radiation`, `chi_so_troi_quang`, `cloud_x_shortwave`...) nên không biết trước
những ngày mây hay thời tiết bất thường. Sai số của Prophet vì thế cao hơn hẳn LightGBM.

Chênh lệch giữa hai mô hình là **bằng chứng định lượng cho giá trị của việc đưa đặc trưng
thời tiết và bước downscale bức xạ vào mô hình**, thay vì chỉ dựa vào tính chu kỳ thời gian.

**Giới hạn diễn giải:** Skill Score đo mức cải thiện so với *một* mô hình đối chứng cụ thể,
không phải so với mọi phương pháp khả dĩ. Một baseline yếu hơn sẽ cho Skill Score cao hơn
mà không phản ánh năng lực mô hình tốt hơn. Con số này không được đọc là bằng chứng cho
tính tối ưu của mô hình.

## 8. Kiểm chứng Toàn vẹn Dữ liệu (Data QA/QC)

In [8]:
kiem_tra = []
for h in HORIZONS:
    m = mask_cham[h]
    co_prophet = m & df_audit[f'prophet_h{h}'].notna()
    kiem_tra.append({
        'horizon': f'h{h}',
        'dòng trong phạm vi': int(m.sum()),
        'dòng có dự báo Prophet': int(co_prophet.sum()),
        'thiếu (%)': round((1 - co_prophet.sum() / m.sum()) * 100, 4),
        'Prophet âm': int((df_audit.loc[co_prophet, f'prophet_h{h}'] < 0).sum()),
        'Prophet vô cực/NaN': int(
            (~np.isfinite(df_audit.loc[co_prophet, f'prophet_h{h}'])).sum()),
    })

df_qa = pd.DataFrame(kiem_tra)
display(df_qa)

assert (df_qa['Prophet âm'] == 0).all(), 'Có dự báo âm — sản lượng không thể âm'
assert (df_qa['Prophet vô cực/NaN'] == 0).all(), 'Có giá trị vô cực/NaN trong dự báo'
print('\nQA/QC đạt: không có dự báo âm, không có giá trị vô cực hay NaN.')

,horizon,dòng trong phạm vi,dòng có dự báo Prophet,thiếu (%),Prophet âm,Prophet vô cực/NaN
0,h1,220114,220114,0.0,0,0
1,h4,202908,202908,0.0,0,0



QA/QC đạt: không có dự báo âm, không có giá trị vô cực hay NaN.


### Nhận xét

Cột *thiếu (%)* cho biết tỷ lệ dòng trong phạm vi chấm điểm mà Prophet không dự báo được
— thường do trạm đó có quá ít dữ liệu lịch sử để học. Tỷ lệ này cần gần 0; nếu lớn thì
Skill Score đang được tính trên một tập con nhỏ hơn dự kiến và phải ghi rõ khi báo cáo.

## 9. Export Processed Dataset

In [9]:
# Bảng theo từng trạm
df_ket_qua.to_csv(OUTPUT_DIR / 'prophet_test_by_site.csv', index=False)

# Tóm tắt Skill Score
if site_loi:
    tom_tat['site_loi'] = site_loi
(OUTPUT_DIR / 'prophet_test_summary.json').write_text(
    json.dumps(tom_tat, ensure_ascii=False, indent=2), encoding='utf-8')

# Dự báo Prophet theo từng dòng — dashboard đọc file này để vẽ đường đối chứng
cot_prophet = [c for c in df_audit.columns if c.startswith('prophet_h')]
df_audit[['site_id', 'timestamp', *cot_prophet]].to_parquet(
    OUTPUT_DIR / 'prophet_test_predictions.parquet', index=False)

# Bảng báo cáo gọn để dán vào LaTeX
df_bao_cao.to_csv(OUTPUT_DIR / 'prophet_bang_bao_cao.csv', index=False)

print('Đã ghi 4 tệp:')
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {p.name:38s} {p.stat().st_size / 1024:8.1f} KB')

Đã ghi 4 tệp:
  prophet_bang_bao_cao.csv                    0.4 KB
  prophet_test_by_site.csv                    2.1 KB
  prophet_test_predictions.parquet         5560.4 KB
  prophet_test_summary.json                   0.4 KB
